In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
from torchvision.transforms.v2.functional import to_pil_image

%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import torch.nn.functional as F
from common import *

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors_darker = sns.color_palette("dark", 5 * 2)
colors = sns.color_palette("deep", 5 * 2)
colors

In [ ]:
import os


def load_from_csv(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            output.append(float(line.strip()))
    return np.array(output)


def load_from_csv_split(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            split = line.strip().split(',')
            if len(output) == 0:
                for i in range(len(split)):
                    output.append([])
            for i in range(len(split)):
                output[i].append(float(split[i].strip()))
    for i in range(len(output)):
        output[i] = np.array(output[i])
    return output

In [ ]:
ratios = [0.05, 0.075, 0.1, 0.2]
loss = []
acc_object = []
acc_background = []
steps = []
eig_list = []
method = "waterbirds_b"
directory = f"/home/gpuadmin/jhkim/feature_preference/practical/waterbirds/datas/BarlowTwins_single_background_0.05"
for r in ratios:
    loss_array = load_from_csv(f"{directory}/{r}/loss.csv")
    [acc_object_array, acc_background_array, step_array] = load_from_csv_split(f"{directory}/{r}/accuracy.csv")
    eigvals_lists_array = load_from_csv_split(f"{directory}/{r}/eig.csv")
    loss.append(loss_array)
    acc_object.append(acc_object_array)
    acc_background.append(acc_background_array)
    steps.append(step_array)
    eig_list.append(eigvals_lists_array)

In [ ]:
background_predictability = 70

fig, axs = plt.subplots(1, 1, figsize=(13, 5.3))

positions = []
plateau_range = 3
for r in range(len(ratios)):
    pos = np.where(acc_object[r] > background_predictability + plateau_range)
    if len(pos[0]) > 0:
        positions.append(pos[0][0])
        axs.axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
        axs.axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
    axs.plot(steps[r], acc_object[r], alpha=0.6, linewidth=5, color=colors_darker[r],
             label=f"Object ratio : {ratios[r]:.3f}")

upper_y = np.zeros_like(steps[0]) + background_predictability + plateau_range
lower_y = np.zeros_like(steps[0]) + background_predictability - plateau_range
axs.fill_between(steps[0], upper_y, lower_y, color='skyblue', alpha=0.4)
axs.axhline(background_predictability, linestyle="--", color="black", linewidth=2, alpha=0.4)
axs.set_xlabel("Step")
axs.set_ylabel("Linear Probing Accuracy")
margin_x = 500 * 0.02
margin_y = 20 * 0.1
axs.set_xlim(0, 500)
axs.set_ylim(65 - margin_y, 80 + margin_y)
axs.text(320, 68.5, "background-label correlation")

axs.legend(ncols=2, loc='lower right')

plt.tight_layout()
plt.savefig(f'/home/gpuadmin/jhkim/feature_preference/practical/waterbirds/figure/{method}.pdf', format='pdf')
plt.show()

In [ ]:
# Threshold sensitivity + normalized AUC for Waterbirds

import numpy as np
import pandas as pd

BASELINE = 70.0
PERSIST = 1

# Original definition is 70 ± 3.
# Sweep the band width around it.
PLATEAU_RANGES = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]

# Same AUC convention as CMNIST
AUC_START_STEP = 200


def first_persistent_crossing(acc, step, threshold, persist=1):
    """First step where accuracy stays >= threshold for `persist` evaluations."""
    acc = np.asarray(acc)
    step = np.asarray(step)

    above = acc >= threshold

    if persist == 1:
        idx = np.where(above)[0]
        return step[idx[0]] if len(idx) > 0 else np.nan

    for i in range(len(above) - persist + 1):
        if np.all(above[i:i + persist]):
            return step[i]

    return np.nan


# ---------------------------------------------------------
# 1. Threshold sensitivity
# ---------------------------------------------------------
threshold_rows = []

for band in PLATEAU_RANGES:
    threshold = BASELINE + band

    escape_times = [
        first_persistent_crossing(
            acc_object[i],
            steps[i],
            threshold=threshold,
            persist=PERSIST,
        )
        for i in range(len(ratios))
    ]

    valid = np.asarray(escape_times, dtype=float)
    valid = valid[~np.isnan(valid)]

    if len(valid) == len(ratios) and np.min(valid) > 0:
        slow_fast_ratio = np.max(valid) / np.min(valid)
    else:
        slow_fast_ratio = np.nan

    # Larger object ratio should escape earlier:
    # escape time should be non-increasing as object ratio increases.
    finite_times = np.asarray(escape_times, dtype=float)
    order_ok = (
        np.all(np.isfinite(finite_times))
        and np.all(np.diff(finite_times) <= 0)
    )

    row = {
        "definition": f"70 ± {band:g}",
        "threshold": threshold,
    }
    row.update({
        f"{r:.3f}": t
        for r, t in zip(ratios, escape_times)
    })
    row["ratio"] = slow_fast_ratio
    row["order"] = "OK" if order_ok else "CHECK"

    threshold_rows.append(row)

threshold_df = pd.DataFrame(threshold_rows)

print("Threshold sensitivity")
display(threshold_df)


# ---------------------------------------------------------
# 2. Normalized AUC above the 70% spurious baseline
# ---------------------------------------------------------
def normalized_auc_above_baseline(
    acc,
    step,
    baseline=70.0,
    start_step=200,
):
    acc = np.asarray(acc, dtype=float)
    step = np.asarray(step, dtype=float)

    mask = step >= start_step
    acc = acc[mask]
    step = step[mask]

    if len(step) < 2:
        return np.nan

    # Only credit performance above the spurious baseline.
    excess_acc = np.maximum(acc - baseline, 0.0)

    auc = np.trapezoid(excess_acc, step)

    # Ideal curve reaches 100% immediately.
    ideal_excess = 100.0 - baseline
    ideal_auc = ideal_excess * (step[-1] - step[0])

    return auc / ideal_auc if ideal_auc > 0 else np.nan


auc_rows = []

for i, r in enumerate(ratios):
    auc = normalized_auc_above_baseline(
        acc_object[i],
        steps[i],
        baseline=BASELINE,
        start_step=AUC_START_STEP,
    )

    auc_rows.append({
        "Object ratio": r,
        "Normalized AUC (%)": 100 * auc,
    })

auc_df = pd.DataFrame(auc_rows)

print("\nNormalized AUC")
display(auc_df)